# Research Notebook

This notebook uses `SignalDenoiserSDK` as the only project-level entry point.

## Project Import Bootstrap
Resolve the project root from the `notebooks/` directory and add it to `sys.path` so imports work cleanly in VS Code.

## Resolve `config.py` Path
Build an absolute path to `config.py` with `pathlib`.

## Initialize `SignalDenoiserSDK`
Instantiate the SDK with the resolved config path and confirm the stored path.

## Run `sdk.prepare_data()`
Generate and partition the dataset through the SDK layer.

## Inspect Dataset Partitions
Review the available split keys and sample counts.

## Validate Split Sizes and Tensor Shapes
Assert the expected 70/15/15 split and verify `(14,)` inputs with `(10,)` targets.

In [1]:
import sys
from pathlib import Path

candidate_paths = [
    Path.cwd() / "notebooks" / "analysis.ipynb",
    Path.cwd() / "analysis.ipynb",
]
NOTEBOOK_PATH = next((path for path in candidate_paths if path.exists()), candidate_paths[0])
PROJECT_ROOT = NOTEBOOK_PATH.resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root added to sys.path: {PROJECT_ROOT}")

Project root added to sys.path: C:\Users\itaym\Documents\HW-ai-orchestration\sine_wave


In [2]:
CONFIG_PATH = PROJECT_ROOT / "src" / "shared" / "config.py"
print(f"Resolved config path: {CONFIG_PATH}")
assert CONFIG_PATH.exists(), f"Missing config.py at {CONFIG_PATH}"

Resolved config path: C:\Users\itaym\Documents\HW-ai-orchestration\sine_wave\src\shared\config.py


In [3]:
from src.sdk.interface import SignalDenoiserSDK

sdk = SignalDenoiserSDK(config_path=str(CONFIG_PATH))
print(f"SDK config path: {sdk.config_path}")

SDK config path: C:\Users\itaym\Documents\HW-ai-orchestration\sine_wave\src\shared\config.py


In [4]:
dataset_splits = sdk.prepare_data()
print("Dataset preparation complete.")

Dataset preparation complete.


In [5]:
print("Available partitions:", list(dataset_splits.keys()))
for split_name, split_data in dataset_splits.items():
    print(
        f"{split_name}: inputs={split_data['inputs'].shape}, targets={split_data['targets'].shape}"
    )

Available partitions: ['train', 'validation', 'test']
train: inputs=(42000, 14), targets=(42000, 10)
validation: inputs=(9000, 14), targets=(9000, 10)
test: inputs=(9000, 14), targets=(9000, 10)


In [6]:
expected_counts = {"train": 42000, "validation": 9000, "test": 9000}
for split_name, expected_count in expected_counts.items():
    split_inputs = dataset_splits[split_name]["inputs"]
    split_targets = dataset_splits[split_name]["targets"]
    assert split_inputs.shape[0] == expected_count, (
        f"{split_name} input count mismatch: {split_inputs.shape[0]} != {expected_count}"
    )
    assert split_targets.shape[0] == expected_count, (
        f"{split_name} target count mismatch: {split_targets.shape[0]} != {expected_count}"
    )
    assert split_inputs.shape[1:] == (14,), (
        f"{split_name} input sample shape mismatch: {split_inputs.shape[1:]}"
    )
    assert split_targets.shape[1:] == (10,), (
        f"{split_name} target sample shape mismatch: {split_targets.shape[1:]}"
    )

print("Split sizes and sample shapes validated successfully.")

Split sizes and sample shapes validated successfully.
